# TA-5：RoBERTa 情感微调（Colab）

**运行前请确认（按需改下一格里的变量）：**

1. `REPO_URL`：已默认 `KevinL35/RSA`；换仓库时再改第二格。
2. `DRIVE_SPLITS`：Google Drive 里三个 CSV 所在目录（已默认 `.../RSA/finetune/data/splits`）。若你的数据在别的路径，只改第二格。

**数据文件：** `train.csv`、`val.csv`、`test.csv`，列需含 `analysis_input_en`、`label_sentiment`（0/1/2）。

**运行时：** 菜单「运行时 → 更改运行时类型 → GPU」建议选 T4 或更好。

**每次训练前：** 请在电脑上 **`git push`** 后，在 Colab **务必运行「克隆 / pull」那一格**（已有 `RSA` 会执行 `git pull`），否则会用过期脚本。

In [ ]:
# ============ 按你的实际情况修改 ============
REPO_URL = "https://github.com/KevinL35/RSA.git"
DRIVE_SPLITS = "/content/drive/MyDrive/RSA/finetune/data/splits"  # 你的三个 CSV 在 Drive 上的目录
# ==========================================

import os
REPO_DIR = "/content/RSA"

## 1) 挂载 Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## 2) 拉取最新代码（首次 clone，以后每次 `git pull`）

In [ ]:
import os
import subprocess

# 默认与上一格「配置」一致；若跳过配置格单独跑本格，也能 clone/pull。
REPO_URL = globals().get("REPO_URL", "https://github.com/KevinL35/RSA.git")
REPO_DIR = globals().get("REPO_DIR", "/content/RSA")

if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)

subprocess.run(["git", "-C", REPO_DIR, "log", "-1", "--oneline"])
print("当前代码版本 ↑  应与 GitHub 最新 commit 一致。OK:", REPO_DIR)

## 3) 安装依赖

In [ ]:
%cd /content/RSA
!pip install -q -r ml/requirements-finetune.txt

## 4) 把 Drive 里的划分文件拷到仓库 `ml/data/splits/`

若你**不用 Drive**、而是把三个 CSV 手动上传到 Colab 左侧并放到 `RSA/ml/data/splits/`，可**跳过本格**，直接跑下一格检查。

In [ ]:
import shutil

dst = os.path.join(REPO_DIR, "ml/data/splits")
os.makedirs(dst, exist_ok=True)

for name in ("train.csv", "val.csv", "test.csv"):
    src = os.path.join(DRIVE_SPLITS, name)
    if not os.path.isfile(src):
        raise FileNotFoundError(f"找不到: {src} 请检查 DRIVE_SPLITS 或文件名")
    shutil.copy2(src, os.path.join(dst, name))
    print("copied", name)

!ls -la {dst}

## 5) 确认文件存在

In [ ]:
!ls -la /content/RSA/ml/data/splits/

## 6) 训练（TA-5）

**建议分两趟：**

1. **先试跑（几分钟级 map）**：下一格设 `TRIAL = True`，用 `--max-train-rows` 等子集，确认 loss/eval 正常、不 OOM。试跑**不会写入** Drive 上的 tokenized 缓存，避免和全量混用。
2. **再全量**：同一格设 `TRIAL = False`，保留 `--tokenized-cache-dir`；第一次会长时间 map，之后同目录可秒开。

若 **`Loading weights 100%` 之后立刻出现 `^C`**，多为 **内存不够**。请先 **`git pull`**，再试 **「高 RAM」** 或保持试跑子集（见 `ml/docs/colab-minimal-start.md`）。

In [ ]:
%cd /content/RSA
import subprocess

# 先试跑再全量：True = 只取前 N 行（不写 tokenized 缓存）；False = 全量 + Drive 缓存
TRIAL = True
CACHE = "/content/drive/MyDrive/RSA/ml_tokenized_cache"

cmd = [
    "python", "ml/scripts/train_sentiment.py",
    "--config", "ml/configs/train_roberta_colab.yaml",
    "--tokenized-cache-dir", CACHE,
]
if TRIAL:
    cmd += [
        "--max-train-rows", "20000",
        "--max-val-rows", "2000",
        "--max-test-rows", "2000",
    ]
subprocess.run(cmd, check=True)

## 7)（可选）仅在测试集上再评一次

In [ ]:
%cd /content/RSA
!python ml/scripts/evaluate_sentiment.py \
  --config ml/configs/train_roberta_colab.yaml \
  --checkpoint_dir ml/artifacts/roberta-sentiment-v0-colab

## 8)（建议）把模型产物拷回 Drive，避免断开连接后丢失

可改 `BACKUP` 目录名。

In [ ]:
import os
import shutil

BACKUP = "/content/drive/MyDrive/RSA/ml_artifacts_colab"
os.makedirs(BACKUP, exist_ok=True)
art = "/content/RSA/ml/artifacts/roberta-sentiment-v0-colab"
dst_art = os.path.join(BACKUP, "roberta-sentiment-v0-colab")
if os.path.isdir(dst_art):
    shutil.rmtree(dst_art)
shutil.copytree(art, dst_art)
rep = "/content/RSA/ml/reports/sentiment_eval_v0_colab.json"
if os.path.isfile(rep):
    shutil.copy2(rep, BACKUP)
print("备份到:", BACKUP)